# 03 — Domino, Manques & Enrichissement GeoJSON

Ce notebook ajoute 3 blocs de données :
1. **Enrichissement GeoJSON** : région et densité (prérequis jumeaux)
2. **Spécialités manquantes** : croisement RPPS × pathologies départementales
3. **Projection domino** : risque de départ à la retraite des médecins

In [1]:
import os as _os
import pandas as pd
import numpy as np
import json
import io
import urllib.request
import re

# Path detection
_cwd = _os.getcwd()
_here = _os.path.dirname(_os.path.abspath("__file__"))

if _os.path.isdir(_os.path.join(_cwd, "notebooks", "data")):
    PROCESSED = _os.path.join(_cwd, "notebooks", "data", "processed")
    RAW = _os.path.join(_cwd, "notebooks", "data", "raw")
elif _os.path.isdir(_os.path.join(_cwd, "data")):
    PROCESSED = _os.path.join(_cwd, "data", "processed")
    RAW = _os.path.join(_cwd, "data", "raw")
else:
    PROCESSED = _os.path.join(_here, "..", "data", "processed")
    RAW = _os.path.join(_here, "..", "data", "raw")

print(f"PROCESSED: {PROCESSED}")
print(f"RAW: {RAW}")

df = pd.read_parquet(f"{PROCESSED}/communes_master.parquet")
print(f"Shape: {df.shape}")
assert 'score' in df.columns, "Score column missing — run 02_score.ipynb first"
print(f"Score column present: {df['score'].notna().sum()} scored communes")


PROCESSED: /Users/zakariyahamdouchi/Dev/Hackathon/.claude/worktrees/agent-a12bb000/notebooks/data/processed
RAW: /Users/zakariyahamdouchi/Dev/Hackathon/.claude/worktrees/agent-a12bb000/notebooks/data/raw
Shape: (34969, 36)
Score column present: 34815 scored communes


## 2. Enrichissement GeoJSON : région et superficie

Jointure avec le GeoJSON Admin Express pour récupérer le code région et la superficie.
La densité (hab/km²) est un prérequis pour le matching de communes jumeaux (TWIN-01).

In [2]:
# Load GeoJSON
with open(f"{RAW}/communes_geo.geojson", 'r') as f:
    geo = json.load(f)

geo_df = pd.DataFrame([feat['properties'] for feat in geo['features']])
print(f"GeoJSON features: {len(geo_df)}")
print(f"GeoJSON columns: {list(geo_df.columns)}")

# Select and rename
geo_cols = geo_df[['codgeo', 'reg', 'superf']].copy()
geo_cols.rename(columns={'codgeo': 'code_commune', 'reg': 'region', 'superf': 'surface_ha'}, inplace=True)
geo_cols['surface_ha'] = pd.to_numeric(geo_cols['surface_ha'], errors='coerce')

# Left join
df = df.merge(geo_cols, on='code_commune', how='left')

# Compute density
df['densite_hab_km2'] = df['population'] / (df['surface_ha'] / 100)
df.loc[df['surface_ha'] == 0, 'densite_hab_km2'] = np.nan
df.loc[df['surface_ha'].isna(), 'densite_hab_km2'] = np.nan

print(f"Region coverage: {df['region'].notna().sum()} / {len(df)}")
print(f"Density coverage: {df['densite_hab_km2'].notna().sum()} / {len(df)}")

assert df['region'].notna().sum() > 34000, f"Region join failed: only {df['region'].notna().sum()}"
assert df['densite_hab_km2'].notna().sum() > 34000, f"Density failed: only {df['densite_hab_km2'].notna().sum()}"


GeoJSON features: 34879
GeoJSON columns: ['codgeo', 'libgeo', 'population', 'can', 'dept', 'reg', 'superf']
Region coverage: 34877 / 34969
Density coverage: 34876 / 34969


## 3. Détection des spécialités manquantes

Croisement RPPS × pathologies départementales : pour chaque commune ayant des données RPPS détaillées, on identifie les spécialités absentes alors que la prévalence départementale est supérieure à la médiane nationale.

| Pathologie | Spécialité attendue |
|---|---|
| prev_diabete | endocrinologue |
| prev_cardio | cardiologue |
| prev_psy | psychiatre |
| prev_cancers | oncologue |
| prev_respiratoire | pneumologue |

In [3]:
PATHO_SPECIALTY_MAP = {
    'prev_diabete': 'endocrinologue',
    'prev_cardio': 'cardiologue',
    'prev_psy': 'psychiatre',
    'prev_cancers': 'oncologue',
    'prev_respiratoire': 'pneumologue',
}

# National medians for pathology prevalence
patho_medians = {}
for col in PATHO_SPECIALTY_MAP.keys():
    patho_medians[col] = df[col].median()
print("Pathology national medians:")
for k, v in patho_medians.items():
    print(f"  {k}: {v:.4f}")

# Compute manques for each commune
manques_list = []
for idx, row in df.iterrows():
    spec_detail = row['specialistes_detail']
    if pd.isna(spec_detail) or spec_detail is None:
        manques_list.append(None)
        continue
    
    # Parse JSON if string
    if isinstance(spec_detail, str):
        try:
            spec = json.loads(spec_detail)
        except (json.JSONDecodeError, ValueError):
            manques_list.append(None)
            continue
    elif isinstance(spec_detail, dict):
        spec = spec_detail
    else:
        manques_list.append(None)
        continue
    
    manques = []
    for patho_col, specialty in PATHO_SPECIALTY_MAP.items():
        dept_prev = row[patho_col]
        if pd.notna(dept_prev) and dept_prev > patho_medians[patho_col]:
            if spec.get(specialty, 0) == 0:
                manques.append(specialty)
    manques_list.append(manques)

df['manques'] = manques_list

has_detail = df['specialistes_detail'].notna().sum()
has_manques_list = sum(1 for m in manques_list if m is not None)
has_gaps = sum(1 for m in manques_list if m is not None and len(m) > 0)
print(f"\nCommunes with specialistes_detail: {has_detail}")
print(f"Communes with manques assessed: {has_manques_list}")
print(f"Communes with at least 1 missing specialty: {has_gaps}")


Pathology national medians:
  prev_diabete: 6.8350
  prev_cardio: 1.9181
  prev_psy: 1.3956
  prev_cancers: 1.3867
  prev_respiratoire: 6.3660



Communes with specialistes_detail: 4039
Communes with manques assessed: 4039
Communes with at least 1 missing specialty: 2756


## 4. Téléchargement DREES : effectifs médecins par tranche d'âge

Source : DREES, démographie des professionnels de santé depuis 2012.
On extrait le % de médecins 55+ par département pour estimer le risque domino.

In [4]:
DREES_URL = (
    "https://data.drees.solidarites-sante.gouv.fr/api/explore/v2.1/catalog/datasets/"
    "la-demographie-des-professionnels-de-sante-depuis-2012/attachments/"
    "medecins_rpps_2012_2025_xlsx"
)

try:
    req = urllib.request.Request(DREES_URL, headers={"User-Agent": "Mozilla/5.0"})
    data = urllib.request.urlopen(req, timeout=120).read()
    print(f"Downloaded DREES file: {len(data) / 1_000_000:.1f} MB")
    
    df_drees = pd.read_excel(io.BytesIO(data), sheet_name='Effectifs', header=0)
    print(f"DREES shape: {df_drees.shape}")
    print(f"Columns: {list(df_drees.columns[:10])}...")
    
    # Check if first row is actual header
    if df_drees.columns[0] != 'territoire' and str(df_drees.iloc[0, 0]).lower().startswith('terr'):
        df_drees.columns = df_drees.iloc[0]
        df_drees = df_drees.iloc[1:].reset_index(drop=True)
        print("Fixed header from first data row")
    
    # Normalize column names
    df_drees.columns = [str(c).strip().lower().replace(' ', '_') for c in df_drees.columns]
    print(f"Normalized columns: {list(df_drees.columns[:10])}...")
    
    drees_downloaded = True
except Exception as e:
    print(f"DREES download failed: {e}")
    print("Using national average fallback (43%)")
    drees_downloaded = False


Downloaded DREES file: 71.2 MB


DREES shape: (309592, 22)
Columns: ['territoire', 'region', 'departement', 'specialites_agregees', 'specialites', 'exercice', 'tranche_age', 'sexe', 'effectif_2012', 'effectif_2013']...
Normalized columns: ['territoire', 'region', 'departement', 'specialites_agregees', 'specialites', 'exercice', 'tranche_age', 'sexe', 'effectif_2012', 'effectif_2013']...


In [5]:
if drees_downloaded:
    print("Available columns:", list(df_drees.columns))
    
    dept_col = 'departement'
    sexe_col = 'sexe'
    spec_col = 'specialites_agregees'
    exercice_col = 'exercice'
    age_col = 'tranche_age'
    eff_col = 'effectif_2025'
    
    # Filter: all sexes, all specialties, all exercise types, exclude national total
    filtered = df_drees[
        (df_drees[sexe_col].astype(str).str.startswith('0-')) &
        (df_drees[spec_col].astype(str).str.startswith('00-')) &
        (df_drees[exercice_col].astype(str).str.startswith('0-')) &
        (~df_drees[dept_col].astype(str).str.startswith('000'))
    ].copy()
    
    print(f"Filtered rows: {len(filtered)}")
    filtered[eff_col] = pd.to_numeric(filtered[eff_col], errors='coerce')
    
    # Total by dept
    total_mask = filtered[age_col].astype(str).str.startswith('00-')
    total_by_dept = filtered[total_mask].set_index(dept_col)[eff_col]
    
    # 55+ tranches: 07-55-59, 08-60-64, 09-65-69, 10-70-74, 11-75+
    aged_mask = filtered[age_col].astype(str).str.match(r'^(07|08|09|10|11)')
    aged_55plus = filtered[aged_mask].groupby(dept_col)[eff_col].sum()
    
    pct_55plus_raw = (aged_55plus / total_by_dept).dropna()
    print(f"\nDepts with 55+ data: {len(pct_55plus_raw)}")
    print(f"National avg 55+: {pct_55plus_raw.mean():.1%}")
    print(f"Range: [{pct_55plus_raw.min():.1%}, {pct_55plus_raw.max():.1%}]")
    
    # Extract clean dept code and match parquet format
    # DREES: '075 -Paris' -> parquet: '75'
    # DREES: '001 -Ain' -> parquet: '01'
    # DREES: '02A -Corse-du-Sud' -> parquet: '2A'
    pct_55plus_by_dept = {}
    for dept_label, pct in pct_55plus_raw.items():
        dept_str = str(dept_label)
        # Extract the code part before the dash
        match = re.match(r'(\d+[AB]?)\s*-', dept_str)
        if match:
            raw_code = match.group(1)
            # For Corse: keep 2A/2B
            if 'A' in raw_code or 'B' in raw_code:
                code = raw_code.lstrip('0')
            else:
                # For numeric codes: match parquet format (01, 02, ..., 09, 10, 11, ...)
                num = int(raw_code)
                code = f"{num:02d}" if num < 100 else str(num)
            pct_55plus_by_dept[code] = pct
    
    print(f"\nMapped dept codes: {len(pct_55plus_by_dept)}")
    # Verify a few match parquet codes
    parquet_depts = set(df['code_departement'].unique())
    matched = set(pct_55plus_by_dept.keys()) & parquet_depts
    print(f"Matched to parquet: {len(matched)} / {len(pct_55plus_by_dept)} DREES depts")
    unmatched_drees = set(pct_55plus_by_dept.keys()) - parquet_depts
    if unmatched_drees:
        print(f"Unmatched DREES codes: {sorted(unmatched_drees)}")
    
    for k in sorted(list(pct_55plus_by_dept.keys()))[:10]:
        print(f"  {k}: {pct_55plus_by_dept[k]:.1%}")
else:
    from collections import defaultdict
    pct_55plus_by_dept = defaultdict(lambda: 0.43)
    print("Using fallback: 43% for all departments")


Available columns: ['territoire', 'region', 'departement', 'specialites_agregees', 'specialites', 'exercice', 'tranche_age', 'sexe', 'effectif_2012', 'effectif_2013', 'effectif_2014', 'effectif_2015', 'effectif_2016', 'effectif_2017', 'effectif_2018', 'effectif_2019', 'effectif_2020', 'effectif_2021', 'effectif_2022', 'effectif_2023', 'effectif_2024', 'effectif_2025']


Filtered rows: 1224

Depts with 55+ data: 102
National avg 55+: 43.3%
Range: [28.8%, 59.2%]

Mapped dept codes: 102
Matched to parquet: 101 / 102 DREES depts
Unmatched DREES codes: ['999']
  01: 45.1%
  02: 48.0%
  03: 47.4%
  04: 58.1%
  05: 35.3%
  06: 43.3%
  07: 48.5%
  08: 51.8%
  09: 45.5%
  10: 48.6%


## 4b. Tendance démographique médecins 2012-2025 (CAGR + delta annuel)

On calcule pour chaque département :
- `effectif_medecins_dept_2025` : effectif total médecins 2025
- `trend_medecins_cagr` : taux de croissance annuel composé 2012→2025 (%)
- `trend_medecins_delta_annuel` : variation annuelle moyenne (nombre de médecins)

In [ ]:
if drees_downloaded:
    # Filter: all sexes, all specialties, all exercise types, all ages (total), exclude national
    trend_filtered = df_drees[
        (df_drees['sexe'].astype(str).str.startswith('0-')) &
        (df_drees['specialites_agregees'].astype(str).str.startswith('00-')) &
        (df_drees['exercice'].astype(str).str.startswith('0-')) &
        (df_drees['tranche_age'].astype(str).str.startswith('00-')) &
        (~df_drees['departement'].astype(str).str.startswith('000'))
    ].copy()
    
    print(f"Trend filtered rows (total all ages, per dept): {len(trend_filtered)}")
    
    # Convert effectif columns to numeric
    year_cols = [f'effectif_{y}' for y in range(2012, 2026)]
    for col in year_cols:
        trend_filtered[col] = pd.to_numeric(trend_filtered[col], errors='coerce')
    
    # Build trend dict per department
    trend_by_dept = {}
    for _, row in trend_filtered.iterrows():
        dept_label = str(row['departement'])
        match = re.match(r'(\d+[AB]?)\s*-', dept_label)
        if not match:
            continue
        raw_code = match.group(1)
        if 'A' in raw_code or 'B' in raw_code:
            code = raw_code.lstrip('0')
        else:
            num = int(raw_code)
            code = f"{num:02d}" if num < 100 else str(num)
        
        eff_2012 = row['effectif_2012']
        eff_2025 = row['effectif_2025']
        
        if pd.notna(eff_2012) and pd.notna(eff_2025) and eff_2012 > 0:
            cagr = ((eff_2025 / eff_2012) ** (1/13) - 1) * 100
            delta = (eff_2025 - eff_2012) / 13
        else:
            cagr = np.nan
            delta = np.nan
        
        trend_by_dept[code] = {
            'effectif_medecins_dept_2025': int(eff_2025) if pd.notna(eff_2025) else np.nan,
            'trend_medecins_cagr': round(cagr, 4) if pd.notna(cagr) else np.nan,
            'trend_medecins_delta_annuel': round(delta, 2) if pd.notna(delta) else np.nan,
        }
    
    print(f"Departments with trend data: {len(trend_by_dept)}")
    
    # Map to communes
    df['effectif_medecins_dept_2025'] = df['code_departement'].map(
        {k: v['effectif_medecins_dept_2025'] for k, v in trend_by_dept.items()}
    )
    df['trend_medecins_cagr'] = df['code_departement'].map(
        {k: v['trend_medecins_cagr'] for k, v in trend_by_dept.items()}
    )
    df['trend_medecins_delta_annuel'] = df['code_departement'].map(
        {k: v['trend_medecins_delta_annuel'] for k, v in trend_by_dept.items()}
    )
    
    # Convert effectif to nullable int
    df['effectif_medecins_dept_2025'] = df['effectif_medecins_dept_2025'].astype('Int64')
    
    print(f"\nCoverage:")
    print(f"  effectif_medecins_dept_2025: {df['effectif_medecins_dept_2025'].notna().sum()} / {len(df)}")
    print(f"  trend_medecins_cagr: {df['trend_medecins_cagr'].notna().sum()} / {len(df)}")
    print(f"  trend_medecins_delta_annuel: {df['trend_medecins_delta_annuel'].notna().sum()} / {len(df)}")
    
    # Sample
    print("\nSample departments:")
    for code in ['75', '13', '69', '78', '59']:
        if code in trend_by_dept:
            t = trend_by_dept[code]
            print(f"  {code}: effectif={t['effectif_medecins_dept_2025']}, CAGR={t['trend_medecins_cagr']:.2f}%, delta={t['trend_medecins_delta_annuel']:.1f}/an")
else:
    df['effectif_medecins_dept_2025'] = pd.array([pd.NA] * len(df), dtype='Int64')
    df['trend_medecins_cagr'] = np.nan
    df['trend_medecins_delta_annuel'] = np.nan
    print("DREES not available — trend columns set to NaN")

## 5. Projection domino

**Méthode :** Le % de médecins 55+ par département (DREES) est appliqué au nombre total de médecins de chaque commune.

**Alerte domino :** déclenchée quand le % 55+ du département dépasse la moyenne nationale OU dépasse 50% (double seuil, D-11).

**Exclusion :** communes avec ≤3 médecins (D-13) — trop peu de données pour un signal fiable.

In [6]:
# Map dept pct_55plus to each commune
df['pct_55plus_dept'] = df['code_departement'].map(pct_55plus_by_dept)

print(f"pct_55plus_dept coverage: {df['pct_55plus_dept'].notna().sum()} / {len(df)}")
missing_depts = df[df['pct_55plus_dept'].isna()]['code_departement'].unique()
if len(missing_depts) > 0:
    print(f"Missing depts ({len(missing_depts)}): {sorted(missing_depts)}")

# Estimate 55+ doctors per commune
df['medecins_55plus_est'] = (df['nb_medecins_total'] * df['pct_55plus_dept']).round(0)

# National dept average
if drees_downloaded:
    avg_dept_pct55 = pct_55plus_raw.mean()
else:
    avg_dept_pct55 = 0.43
print(f"Average dept %55+: {avg_dept_pct55:.1%}")

# Domino alert (double threshold per D-11)
df['domino_alert'] = (df['pct_55plus_dept'] > avg_dept_pct55) | (df['pct_55plus_dept'] > 0.5)
# Convert to nullable boolean to support None
df['domino_alert'] = df['domino_alert'].astype('object')

# Null out for communes with <=3 doctors (D-13)
small_mask = df['nb_medecins_total'] <= 3
df.loc[small_mask, 'domino_alert'] = None
df.loc[small_mask, 'medecins_55plus_est'] = np.nan

# Projection text
def make_projection(row):
    if pd.isna(row['medecins_55plus_est']) or row['medecins_55plus_est'] <= 0:
        return None
    if row['nb_medecins_total'] <= 3:
        return None
    return f"-{int(row['medecins_55plus_est'])} medecins (estimation)"

df['projection_2030'] = df.apply(make_projection, axis=1)

eligible = df[df['nb_medecins_total'] > 3]
print(f"\nEligible communes (>3 doctors): {len(eligible)}")
print(f"Domino alerts: {eligible['domino_alert'].apply(lambda x: x == True).sum()}")
print(f"With projection: {df['projection_2030'].notna().sum()}")


pct_55plus_dept coverage: 34875 / 34969
Missing depts (4): ['975', '977', '978', '98']
Average dept %55+: 43.3%

Eligible communes (>3 doctors): 4934
Domino alerts: 1813
With projection: 4916


## 6. Validation

In [ ]:
assert df['region'].notna().sum() > 34000, f"Region join failed: {df['region'].notna().sum()}"
assert df['densite_hab_km2'].notna().sum() > 34000, f"Density failed: {df['densite_hab_km2'].notna().sum()}"
assert df['pct_55plus_dept'].notna().sum() > 4000, f"DREES dept join failed: {df['pct_55plus_dept'].notna().sum()}"
assert (df[df['nb_medecins_total'] > 3]['domino_alert'].notna()).sum() > 4000, "Domino alert missing for eligible communes"
assert df.loc[df['nb_medecins_total'] <= 3, 'domino_alert'].isna().all(), "Domino alert should be null for <=3 doctor communes"

# Trend columns validation
assert df['effectif_medecins_dept_2025'].notna().sum() > 34000, f"Trend effectif failed: {df['effectif_medecins_dept_2025'].notna().sum()}"
assert df['trend_medecins_cagr'].notna().sum() > 34000, f"Trend CAGR failed: {df['trend_medecins_cagr'].notna().sum()}"
assert df['trend_medecins_delta_annuel'].notna().sum() > 34000, f"Trend delta failed: {df['trend_medecins_delta_annuel'].notna().sum()}"

# Check improved coverage after dept code fix
print(f"pct_55plus_dept coverage: {df['pct_55plus_dept'].notna().sum()} / {len(df)} ({df['pct_55plus_dept'].notna().mean():.1%})")
print(f"trend_medecins_cagr coverage: {df['trend_medecins_cagr'].notna().sum()} / {len(df)} ({df['trend_medecins_cagr'].notna().mean():.1%})")

print("\nALL ASSERTIONS PASSED")
print(f"Final column count: {df.shape[1]}")
print(f"New columns: region, surface_ha, densite_hab_km2, manques, pct_55plus_dept, medecins_55plus_est, domino_alert, projection_2030, effectif_medecins_dept_2025, trend_medecins_cagr, trend_medecins_delta_annuel")

## 7. Sauvegarde

In [8]:
df.to_parquet(f"{PROCESSED}/communes_master.parquet", index=False)
print(f"Saved: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nAll columns: {list(df.columns)}")


Saved: 34969 rows, 44 columns

All columns: ['code_commune', 'nom_commune', 'population', 'code_departement', 'nb_generalistes', 'nb_medecins_total', 'nb_specialistes', 'specialistes_detail', 'apl_2022', 'apl_2023', 'apl', 'apl_evolution', 'taux_pauvrete', 'revenu_median', 'pct_75_plus', 'temps_urgences_min', 'temps_urgences_mcs_min', 'nb_etablissements', 'has_hopital', 'has_ehpad', 'nb_msp', 'has_msp', 'latitude', 'longitude', 'prev_diabete', 'prev_cardio', 'prev_psy', 'prev_cancers', 'prev_respiratoire', 'score', 'n_score_components', 'score_apl_norm', 'score_pauvrete_norm', 'score_pct75_norm', 'score_urgences_norm', 'classe', 'region', 'surface_ha', 'densite_hab_km2', 'manques', 'pct_55plus_dept', 'medecins_55plus_est', 'domino_alert', 'projection_2030']
